# 策略梯度深入解析

## 目录
1.  **引言**
2.  **语言模型的强化学习设定**
3.  **策略梯度方法详解**
    * 目标与梯度推导
    * 挑战：高方差
    * 使用基线 (Baselines) 降低方差
    * 优势函数 (Advantage Functions)
4.  **训练实践：GRPO 算法**
    * GRPO 简介
    * 核心步骤
    * 不同 Delta (δ) 的计算方式
    * 损失函数的计算
5.  **实验与结果**
6.  **总结**

---

## 1. 引言

上一讲我们概述了从可验证奖励中进行强化学习 (RLVR) 的方法，其核心是策略梯度。本次讲座将深入探讨**策略梯度**的内在机制，以 **GRPO (Group Relative Policy Optimization)** 算法为例进行说明。

---

## 2. 语言模型的强化学习设定

在语言模型 (LM) 的情境下，强化学习的各个要素可以定义如下：

* **状态 (State) `s`**: 模型的输入提示 (prompt) 以及到目前为止已生成的响应内容。
* **动作 (Action) `a`**: 生成下一个 token。
* **奖励 (Reward) `R`**: 对整个响应的好坏程度的评估。本讲座主要关注：
    * **结果奖励 (Outcome rewards)**: 仅依赖于最终完整响应的奖励。
    * **可验证奖励 (Verifiable rewards)**: 奖励的计算过程是确定性的。
    * *注意：由于我们评估的是最终结果，传统 RL 中的折扣因子 (discounting) 和自举 (bootstrapping) 概念在这里不太适用。*
* **转移概率 (Transition probabilities) `T(s' | s, a)`**: 在 LM 中，状态转移是**确定性**的，即新状态 `s'` 就是旧状态 `s` 加上新生成的 token `a` (`s' = s + a`)。
* **策略 (Policy) `π(a | s)`**: 语言模型本身，通常是一个经过微调的模型。
* **Rollout/Episode**: 从一个提示 `s` 开始，通过一系列动作 `a` 生成完整响应，并最终获得奖励 `R` 的过程。
* **目标**: 最大化期望奖励 `E[R]`。

---

## 3. 策略梯度方法详解

为简化符号，我们用 `a` 表示整个响应序列。

### 目标与梯度推导

我们的目标是最大化关于策略 `π` 的期望奖励：
$$\mathbb{E}[R] = \int p(s) \pi(a | s) R(s, a)$$

为了通过梯度上升来优化模型参数，我们对期望奖励求导：
$$\nabla \mathbb{E}[R] = \int p(s) \nabla \pi(a | s) R(s, a)$$

这里使用一个技巧，即 $\nabla \pi = \pi \nabla \log \pi$：
$$\nabla \mathbb{E}[R] = \int p(s) \pi(a | s) \nabla \log \pi(a | s) R(s, a)$$

最终得到策略梯度的期望形式：
$$\nabla \mathbb{E}[R] = \mathbb{E}[\nabla \log \pi(a | s) R(s, a)]$$

基于此，**朴素策略梯度 (Naive policy gradient)** 的更新方式是：
1.  采样一个提示 `s` 和一个响应 `a ~ π(a | s)`。
2.  基于 `∇ log π(a | s) R(s, a)` 更新模型参数。

这与监督微调 (SFT) 非常相似，区别在于每个样本的梯度被其奖励 `R(s, a)` 加权了。如果奖励是二元（0 或 1）的，那么模型只会在获得奖励为 1 的正确响应上进行更新。

### 挑战：高方差

策略梯度方法的主要挑战是**梯度估计的噪声/方差非常高**。尤其是在奖励稀疏（大部分响应奖励为 0，只有少数为 1）的情况下，模型需要大量采样才能获得有效的学习信号。这与 RLHF 中使用奖励模型提供更连续、平滑的奖励信号形成对比。

### 使用基线 (Baselines) 降低方差

为了解决高方差问题，我们可以从奖励中减去一个**基线 (baseline)** `b(s)`。这个基线只与状态 `s` 有关，与动作 `a` 无关，因此不会改变梯度的期望值（即梯度是无偏的）。更新规则变为：
$$\text{Update based on } \nabla \log \pi(a | s) (R(s, a) - b(s))$$
通过选择合适的 `b(s)`，可以显著降低梯度估计的方差。例如，如果一个动作的奖励 `R(s, a)` 虽然绝对值高，但低于该状态下的平均水平 `b(s)`，那么 `R(s, a) - b(s)` 将是负数，模型会降低执行该动作的概率。

### 优势函数 (Advantage Functions)

一个常用且有效的基线是**状态价值函数 `V(s)`**，即在状态 `s` 下的期望奖励：
$$b(s) = V(s) = \mathbb{E}[R | s]$$
使用这个基线后，奖励部分 `R(s, a) - V(s)` 正是**优势函数 `A(s, a)`** 的定义：
$$A(s, a) = Q(s, a) - V(s)$$
其中 `Q(s, a)` 是在状态 `s` 下执行动作 `a` 后的期望奖励。优势函数的直观含义是：在状态 `s` 下，采取动作 `a` 比平均水平要好多少。

---

## 4. 训练实践：GRPO 算法

### GRPO 简介
[cite_start]**Group Relative Policy Optimization (GRPO)** [cite: 3] [cite_start]是对 PPO [cite: 2] 的一种简化，它**移除了 Critic (价值函数)**。GRPO 利用了语言模型任务中的一个特性：可以为单个提示生成一组（group）响应，这为计算基线提供了一个自然的方法。

### 核心步骤

GRPO 的训练流程可以概括为以下几步：
1.  **生成响应**: 对于每个提示，使用当前策略模型生成一组（例如 N 个）响应。
2.  **计算奖励**: 对每个生成的响应计算奖励 `R`。
3.  **计算 Delta (δ)**: 根据这组响应的奖励，计算一个类似优势函数的值 `δ`，用于加权梯度。
4.  **计算对数概率**: 计算模型生成这些响应的对数概率 `log π(a | s)`。
5.  **计算损失**: 结合对数概率和 `δ` 计算最终的损失函数，并进行梯度更新。

### 不同 Delta (δ) 的计算方式

`δ` 的计算有多种模式，这些模式本质上是不同类型的基线：
* `rewards`: **不使用基线**，`δ` 就是原始奖励 `R`。
* `centered_rewards`: **使用组内均值作为基线**。`δ = R - mean(R_group)`。这可以惩罚那些低于组内平均水平的响应。
* [cite_start]`normalized_rewards`: **使用 z-score 标准化**。`δ = (R - mean(R_group)) / std(R_group)`。这是 GRPO 论文中提出的方法 [cite: 3]。
* `max_rewards`: 一种启发式方法，只对组内奖励最高的响应给予正向梯度，其他响应的 `δ` 为 0。

### 损失函数的计算

损失函数的计算也有多种模式：
* `naive`: 直接使用 `δ` 和对数概率计算，损失为 `- (log_probs * δ).mean()`。
* `unclipped` / `clipped`: 类似于 PPO，引入旧策略的对数概率 `old_log_probs`，计算概率比率 `ratios = exp(log_probs - old_log_probs)`。`clipped` 模式会对 `ratios` 进行裁剪，防止更新步长过大，从而提高训练稳定性。

*一个重要的技术细节是在计算 `ratios` 时，必须**冻结旧策略的参数** (`old_log_probs`)，避免对其进行求导，否则无法得到正确的梯度。*

---

## 5. 实验与结果

在一个简单的排序任务中，对不同 `δ` 计算方式进行实验，得到以下结论：

* **使用原始奖励 (`rewards`)**: 模型学习效果不佳，难以收敛到最优解。
* **使用中心化奖励 (`centered_rewards`)**: 效果有明显提升。因为低于平均水平的响应会得到负向更新，而当所有响应奖励相同时，梯度为零，避免了无效更新。尽管如此，模型仍可能陷入局部最优。
* **使用标准化奖励 (`normalized_rewards`)**: 在当前固定长度的排序任务中，与中心化奖励相比差别不大。但需要注意的是，有研究指出这种标准化可能引入**长度偏见**，因此像 Dr. GRPO 这样的改进算法选择不进行此项操作。

总的来说，强化学习的调优并非易事，很容易陷入次优状态，超参数的调整至关重要。

---

## 6. 总结

* 强化学习是模型能力**超越人类**的关键，核心思想是 "If you can measure it, you can optimize it"（如果你能衡量它，你就能优化它）。
* 策略梯度框架在概念上清晰明了，但需要**引入基线来降低方差**，从而实现有效学习。
* RL 系统比预训练系统要复杂得多，因为它涉及到**推理和训练的混合负载**，并且需要管理多个模型（策略模型、参考模型等）。